# 第5周 · 分类、评估与交叉验证

**本周结束时你能做到**：训练第一个分类器并画出它的决策边界；在不平衡数据上选对评估指标；用分层交叉验证可靠地比较逻辑回归与随机森林；用偏差-方差分解解释森林为什么比单棵树强，并把梯度提升树加进表格数据的强基线；识别并避免四类数据泄漏。

上周我们预测的是连续数值（回归）；这周开始预测类别（分类）：这位乘客生还了吗？这条影评是好评吗？这张图片是哪个数字？

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import data
from zero2researcher.checks import week05 as ck

---
## S1 · 逻辑回归：把线性输出变成概率

线性回归输出任意实数，但"是不是好评"需要一个 0 到 1 之间的概率。逻辑回归在线性输出 $z = w \cdot x + b$ 外面套了一个 **sigmoid** 函数：

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

它把任意实数压进 (0, 1)：$z$ 很大时输出接近 1，$z$ 很小时接近 0，$z = 0$ 时恰好 0.5。

In [ ]:
def sigmoid(z):
    # 在这里写你的代码
    raise NotImplementedError

z = np.linspace(-8, 8, 200)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(z, sigmoid(z))
ax.axhline(0.5, color="gray", linestyle=":")
ax.set_xlabel("Linear output z")
ax.set_ylabel("sigmoid(z)")
ax.set_title("Sigmoid squashes any number into (0, 1)")
plt.show()

### 2D 数据上的决策边界

造一份二维的两类数据，训练逻辑回归，然后对平面上每个点问模型"你觉得这是哪一类"——涂色出来就是**决策边界**。

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

X2, y2 = make_classification(n_samples=300, n_features=2, n_informative=2, n_redundant=0,
                             n_clusters_per_class=1, class_sep=1.2, random_state=42)
clf = LogisticRegression().fit(X2, y2)

grid_x, grid_y = np.meshgrid(np.linspace(X2[:, 0].min() - 1, X2[:, 0].max() + 1, 200),
                             np.linspace(X2[:, 1].min() - 1, X2[:, 1].max() + 1, 200))
proba = clf.predict_proba(np.c_[grid_x.ravel(), grid_y.ravel()])[:, 1].reshape(grid_x.shape)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
cs = ax.contourf(grid_x, grid_y, proba, levels=20, cmap="RdBu_r", alpha=0.6)
fig.colorbar(cs, label="P(class 1)")
ax.scatter(X2[:, 0], X2[:, 1], c=y2, cmap="RdBu_r", edgecolor="k", s=18)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title("Logistic regression draws a straight decision boundary")
plt.show()

### 交叉熵：对正确答案感到意外的程度

逻辑回归的损失是**交叉熵**：模型给正确类别的概率是 $p$，损失就是 $-\log(p)$。

- 给正确答案 0.99 的概率 → 损失几乎为 0（毫不意外）。
- 给正确答案 0.01 的概率 → 损失巨大（"信誓旦旦地答错"被罚得最狠）。

训练就是不断调整 $w, b$，让模型在全部训练样本上的平均"意外程度"最小。

In [ ]:
p = np.linspace(0.001, 0.999, 300)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(p, -np.log(p))
ax.set_xlabel("Predicted probability for the true class")
ax.set_ylabel("Loss  -log(p)")
ax.set_title("Confidently wrong predictions are punished hardest")
plt.show()
print("预测 0.9 时的损失:", round(-np.log(0.9), 3), "   预测 0.1 时的损失:", round(-np.log(0.1), 3))

### 混淆矩阵与四个指标

把预测和真相做成 2×2 的表，就是**混淆矩阵**（以"1 = 阳性"为约定）：

|  | 预测 1 | 预测 0 |
| --- | --- | --- |
| **真实 1** | TP（真阳性） | FN（假阴性，漏报） |
| **真实 0** | FP（假阳性，误报） | TN（真阴性） |

> 注意：下一格 sklearn 画的混淆矩阵按标签 0、1 的顺序排列——**左上角是 TN、右下角才是 TP**，行列顺序与上表正好相反。读任何混淆矩阵之前，先看清两条坐标轴各是什么。

- **准确率** accuracy = (TP + TN) / 全部 —— 预测对的比例
- **精确率** precision = TP / (TP + FP) —— 报出来的阳性里有多少是真的
- **召回率** recall = TP / (TP + FN) —— 真的阳性里抓住了多少
- **F1** = 精确率与召回率的调和平均；**macro-F1** 是对每一类分别算 F1 再取平均
- **ROC-AUC**：随机取一个正样本和一个负样本，模型给正样本更高分的概率

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, random_state=42, stratify=y2)
clf = LogisticRegression().fit(X2_tr, y2_tr)
pred = clf.predict(X2_te)

ConfusionMatrixDisplay.from_predictions(y2_te, pred)
plt.title("Confusion matrix on the held-out test set")
plt.show()
print(classification_report(y2_te, pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y2_te, clf.predict_proba(X2_te)[:, 1]), 3))

### 准确率什么时候骗人：不平衡数据

现实中的类别经常严重不平衡：罕见病筛查里 95% 的人是健康的。造一份 95:5 的数据看看——一个**什么都不学、永远喊"健康"** 的"模型"，准确率是多少？

（注意 `flip_y=0`：`make_classification` 默认会随机翻转 1% 的标签，会让比例偏离 95:5。）

In [ ]:
Xi, yi = make_classification(n_samples=2000, n_features=2, n_informative=2, n_redundant=0,
                             n_clusters_per_class=1, weights=[0.95, 0.05], flip_y=0,
                             class_sep=1.0, random_state=42)
print("两类的样本数:", np.bincount(yi))

# 在这里写你的代码
raise NotImplementedError
print("什么都不学、全预测多数类，准确率 =", majority_acc)
ck.imbalanced_majority_acc(majority_acc)

In [ ]:
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(Xi, yi, random_state=42, stratify=yi)
clf_i = LogisticRegression().fit(Xi_tr, yi_tr)
print(classification_report(yi_te, clf_i.predict(Xi_te), digits=3))
print("ROC-AUC:", round(roc_auc_score(yi_te, clf_i.predict_proba(Xi_te)[:, 1]), 3))

对照上一格的输出读三个数：

- **准确率 0.950**——和那个什么都不学、永远喊"健康"的基线一模一样；
- 少数类（`1`）的**召回率只有 0.28**：测试集里 25 个真正的阳性只抓到 7 个，漏掉了七成多（多数类的召回率是 0.985）；
- **macro-F1 只有约 0.67**，一眼就暴露了问题。

ROC-AUC 却有约 0.97：模型给阳性打的分整体上确实更高（排序能力不差），坏就坏在默认的 0.5 阈值上。ROC-AUC 不管阈值选得好不好，所以它要和混淆矩阵、macro-F1 一起看。

**结论**：不平衡数据不要只看准确率；看混淆矩阵、macro-F1（它对每一类一视同仁），以及与阈值无关的 ROC-AUC。

---
## S2 · 为什么需要验证集与交叉验证

在训练集上评估自己，就像出题人给自己打分；用测试集反复调参，就像考试前偷看了答案——两者的分数都不可信。标准做法：

- **训练集**：学参数。**验证集**：选模型、调超参数。**测试集**：最后只用一次，报告成绩。
- 数据不多时用 **k 折交叉验证**：切成 k 份，轮流用 k−1 份训练、1 份验证，报告 k 个分数的均值 ± 标准差。
- 分类任务用 **StratifiedKFold**（分层 k 折）：每一折里两类的比例都和整体一致——不平衡数据尤其重要。

### 实战：Titanic 生存预测

用四个特征预测 `survived`：`sex`、`pclass`、`age`（有 177 个缺失）、`fare`。

In [ ]:
titanic = data.load_titanic()
X = titanic[["pclass", "age", "fare"]].copy()
X["sex"] = (titanic["sex"] == "female").astype(int)   # female=1, male=0
y = titanic["survived"]
print(X.isna().sum())
X.head()

### 预处理必须放进 Pipeline

"用中位数填充缺失"与"标准化"都是**从数据里学出来的步骤**（中位数、均值、标准差都是参数！）。验证折和测试集扮演的是"未来的数据"——真实使用时，你不可能提前拿到它们来算中位数。所以如果先在全部数据上算好、再做交叉验证，验证折的信息就提前泄漏给了训练。Kaggle 的数据泄漏教程把话说得很直白：*"exclude the validation data from any type of fitting, including the fitting of preprocessing steps"*；scikit-learn 文档的 Common pitfalls 一节给出的总规则也是：永远不要在测试数据上调用 `fit`。

做法：把 `SimpleImputer`（中位数填充）→ `StandardScaler` → `LogisticRegression` 打包成一个 **Pipeline**，再把整个 Pipeline 交给交叉验证——每一折内部自动只用训练折去 fit 预处理。**这是本课程从今天起的标准写法。**

自查函数会同时检查你交给交叉验证的 Pipeline 和 `X`：填充与标准化必须在 Pipeline 里，`X` 必须是还带着缺失值的原始特征。

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate

# 在这里写你的代码
raise NotImplementedError
print("5 折准确率:", np.round(scores, 3))
print(f"平均 {scores.mean():.3f} ± {scores.std():.3f}")
ck.titanic_cv_scores(scores, lr_pipe, X)

约 0.79 ± 0.01。注意**标准差和均值一样重要**：两个模型差 0.5 个百分点、而折间波动有 3 个百分点时，这个差距什么都说明不了。

### 学习曲线：更多数据有用吗？

把训练集从小到大，画"训练分数 vs 验证分数"两条曲线：

In [ ]:
from sklearn.model_selection import learning_curve

sizes, tr_sc, va_sc = learning_curve(lr_pipe, X, y, cv=cv,
                                     train_sizes=np.linspace(0.1, 1.0, 6),
                                     shuffle=True, random_state=42)
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(sizes, tr_sc.mean(axis=1), "o-", label="train")
ax.plot(sizes, va_sc.mean(axis=1), "o-", label="validation")
ax.set_xlabel("Training set size")
ax.set_ylabel("Accuracy")
ax.set_title("Both curves plateau: more data alone won't help much")
ax.legend()
plt.show()

读图口诀：
- 两条线都低、且已贴在一起 → **欠拟合**，换更强的模型或更好的特征，加数据没用。
- 训练线高、验证线低、缺口大 → **过拟合**，加数据或加正则化有望缩小缺口。

### 验证曲线：正则化强度 C

逻辑回归用参数 `C` 控制正则化强度：**C 越小，正则化越强**（模型被压得越"平"）。

> ⚠️ sklearn **1.8** 起 `LogisticRegression` 的 `penalty` 参数已弃用（1.10 将移除）。旧教程里的 `penalty="l2"` 写法不要再用：正则化的**强度**用 `C` 调，正则化的**类型**用 `l1_ratio` 选——`l1_ratio=0`（默认）是 L2，`l1_ratio=1` 是 L1，介于两者之间是弹性网。默认的 `lbfgs` 求解器只支持 L2；要用 L1 得换 `solver="liblinear"` 或 `"saga"`，弹性网只有 `"saga"` 支持。
>
> L1 和 L2 的直观区别：**L2** 把所有权重整体往 0 收缩，但很少恰好变成 0；**L1** 会把不重要特征的权重直接压成 0，相当于顺带做了特征选择——在基因表达这类"特征多、样本少"的数据上很常用。本课程默认用 L2。

In [ ]:
Cs = [0.001, 0.01, 0.1, 1, 10, 100]
tr_means, va_means = [], []
for C in Cs:
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         LogisticRegression(C=C, max_iter=1000))
    res = cross_validate(pipe, X, y, cv=cv, return_train_score=True)
    tr_means.append(res["train_score"].mean())
    va_means.append(res["test_score"].mean())

fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogx(Cs, tr_means, "o-", label="train")
ax.semilogx(Cs, va_means, "o-", label="validation")
ax.set_xlabel("C (inverse regularization strength)")
ax.set_ylabel("Accuracy")
ax.set_title("Too much regularization (small C) underfits")
ax.legend()
plt.show()

---
## S3 · 决策树与随机森林

决策树就是"20 个问题"游戏：每个节点问一个是/否问题（"是女性吗？""票价 > 10 吗？"），一路问到叶子，叶子里多数人的命运就是预测。先看一棵浅树长什么样：

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree_pipe = make_pipeline(SimpleImputer(strategy="median"),
                          DecisionTreeClassifier(max_depth=3, random_state=42))
tree_pipe.fit(X, y)

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(tree_pipe[-1], feature_names=X.columns, class_names=["died", "survived"],
          filled=True, fontsize=8, ax=ax)
ax.set_title("A depth-3 decision tree is a flowchart of yes/no questions")
plt.show()

第一个问题就是性别——树自己从数据里学出了"妇女优先"。

单棵深树很容易过拟合（把训练集背下来）。**随机森林**：训练很多棵"各自见过不同的 bootstrap 样本（有放回抽样得到的数据子集），并且每次分裂只在随机抽出的几个特征里挑"的树，预测时投票——个体的过拟合在平均中互相抵消。这叫**集成**（ensemble）。三个最常用的超参数：

- `n_estimators`：树的数量（默认 100，多一般更稳但更慢）
- `max_depth`：每棵树的最大深度（默认不限制）
- `max_features`：每次分裂随机考虑的特征数（默认 `sqrt`，这是"随机"的来源之一）

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 在这里写你的代码
raise NotImplementedError
for name, pipe in [("LogisticRegression", lr_pipe), ("RandomForest", rf_pipe)]:
    s = cross_val_score(pipe, X, y, cv=cv)
    print(f"{name:20s} {s.mean():.3f} ± {s.std():.3f}")

同一套交叉验证下随机森林约 0.83，比逻辑回归高 4 个百分点左右——在同一个 `cv` 上比较才公平。

### 特征重要性及其陷阱

随机森林自带 `feature_importances_`（基于不纯度下降），但它**偏爱取值多的连续特征**（age、fare 有几百种取值，sex 只有 2 种，连续特征有更多机会被选来分裂）。更可信的是 **permutation importance**：在留出的数据上把某一列打乱，看分数掉多少。

In [ ]:
from sklearn.inspection import permutation_importance

X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=42, stratify=y)
rf_pipe.fit(X_tr, y_tr)
imp = rf_pipe[-1].feature_importances_
perm = permutation_importance(rf_pipe, X_te, y_te, n_repeats=10, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(X.columns, imp)
axes[0].set_title("Impurity importance (biased)")
axes[1].bar(X.columns, perm.importances_mean)
axes[1].set_title("Permutation importance (held-out data)")
for a in axes:
    a.set_ylabel("Importance")
plt.tight_layout()
plt.show()

左图说 fare 最重要；右图（打乱后掉分最多的）说 **sex** 才是——与我们对这场灾难的了解一致。读论文看到"特征重要性"时，先问一句：是哪一种？

### 为什么森林比一棵树强：偏差-方差分解

上面说随机森林靠"集体投票"抗过拟合。这句话能量化吗？能，前提是**知道真相**。回到第 4 周的阳性对照思路：自己造数据、真函数已知，就能把一个模型的测试误差精确地拆成三块：

$$\underbrace{\mathbb{E}\big[(y - \hat f(x))^2\big]}_{\text{误差}} = \underbrace{\big(f(x) - \mathbb{E}[\hat f(x)]\big)^2}_{\text{偏差}^2} + \underbrace{\mathrm{Var}\big[\hat f(x)\big]}_{\text{方差}} + \underbrace{\sigma^2}_{\text{噪声}}$$

这里的期望（平均）是对"抽到哪一份训练集"和"测试点上的噪声"取的：

- **偏差²**：用很多份训练集各训一个模型，它们的**平均预测**离真函数 f 有多远——模型系统性地错多少；
- **方差**：这些模型的预测彼此差多少——模型对"碰巧抽到哪份训练集"有多敏感；
- **噪声**：数据本身的随机性 σ²，谁也降不下去的地板。

实验照搬 scikit-learn 的示例 *Single estimator versus bagging: bias-variance decomposition* 的设定：一维输入，真函数 f(x) = exp(−x²) + 1.5·exp(−(x−2)²)，噪声标准差 σ = 0.1；造 50 份训练集、每份 50 个点；测试集 1000 个点，每个点上抽 50 份带噪声的 y。用实验室的话说，就是做 **50 次独立的生物学重复**（每次重新取样、重新训练），再看结果的平均与离散。

In [ ]:
# 设定改写自 scikit-learn 示例 "Single estimator versus bagging: bias-variance decomposition"
# （examples/ensemble/plot_bias_variance.py，BSD-3-Clause，© the scikit-learn developers）：
# 真函数、样本量与噪声照搬；随机数改用 default_rng，分解改写成向量化形式。
# https://scikit-learn.org/stable/auto_examples/ensemble/plot_bias_variance.html
N_REPEAT, N_TRAIN, N_TEST = 50, 50, 1000


def f(x):
    """真函数（只有造数据的人知道）"""
    x = np.ravel(x)
    return np.exp(-x ** 2) + 1.5 * np.exp(-(x - 2) ** 2)


def make_datasets(noise=0.1, seed=0):
    """50 份独立的训练集；以及 1000 个测试点，每个点上 50 份带噪声的 y"""
    rng = np.random.default_rng(seed)
    train_sets = []
    for _ in range(N_REPEAT):
        x = rng.uniform(-5, 5, N_TRAIN)
        train_sets.append((x.reshape(-1, 1), f(x) + rng.normal(0, noise, N_TRAIN)))
    x_test = np.sort(rng.uniform(-5, 5, N_TEST)).reshape(-1, 1)
    y_test = f(x_test)[:, None] + rng.normal(0, noise, (N_TEST, N_REPEAT))
    return train_sets, x_test, y_test


train_sets, X_test_bv, y_test_bv = make_datasets()
f_test = f(X_test_bv)
print(len(train_sets), "份训练集，每份 X 的形状", train_sets[0][0].shape)
print("测试点", X_test_bv.shape, "；带噪声的 y", y_test_bv.shape, "：每个测试点 50 份")

写分解函数 `decompose(y_predict, y_test, f_test)`。三个输入的形状：

- `y_predict`：(1000, 50)，第 j 列是用第 j 份训练集训出的模型在 1000 个测试点上的预测；
- `y_test`：(1000, 50)，每个测试点上 50 份带噪声的观测；
- `f_test`：(1000,)，测试点上的真函数值。

四个量都先在每个测试点上算，再对测试点取平均：

- **误差**：每份带噪声的 y 与每个模型的预测**两两配对**求平方差，全部平均。`y_test[:, :, None] - y_predict[:, None, :]` 的形状是 (1000, 50, 50)，广播一次就把配对做完了；照 sklearn 示例写成两层 `for` 循环也可以（循环上界用数组的形状，例如 `y_predict.shape[1]`，比写死 50 更稳）；
- **偏差²**：先对 50 个模型的预测取平均（`axis=1`），再和**真函数 f**（不是带噪声的 y）比，平方；
- **方差**：50 个预测的方差（`np.var`，默认 `ddof=0`，与 sklearn 示例一致）；
- **噪声**：50 份带噪声 y 的方差。

In [ ]:
def decompose(y_predict, y_test, f_test):
    """返回 (误差, 偏差², 方差, 噪声)，都已对测试点取平均"""
    # 在这里写你的代码
    raise NotImplementedError

ck.bias_variance(decompose)

接下来在 50 份训练集上各训一次模型、收集预测。`make_model(i)` 返回第 i 份训练集要用的**全新**模型（Bagging 和森林内部的随机性用 `random_state=i` 固定，保证可复现）。比较三个模型：

- 单棵**不限深度**的回归树；
- **Bagging**：10 棵树，每棵用一份 bootstrap 重采样的训练数据，预测时取平均；
- **随机森林**：100 棵树。

写 `predict_all(make_model)`，返回形状 (1000, 50) 的预测矩阵：

In [ ]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor


def predict_all(make_model):
    """在 50 份训练集上各训一个新模型，返回 (1000, 50) 的预测矩阵"""
    # 在这里写你的代码
    raise NotImplementedError


models_bv = {
    "Tree": lambda i: DecisionTreeRegressor(random_state=0),
    "Bagging(Tree)": lambda i: BaggingRegressor(DecisionTreeRegressor(), n_estimators=10, random_state=i),
    "RandomForest": lambda i: RandomForestRegressor(n_estimators=100, random_state=i),
}
results_bv, preds_bv = {}, {}
for name, make_model in models_bv.items():
    preds_bv[name] = predict_all(make_model)
    results_bv[name] = decompose(preds_bv[name], y_test_bv, f_test)
    e, b2, v, n = results_bv[name]
    print(f"{name:14s} {e:.4f}（误差）= {b2:.4f}（偏差²）+ {v:.4f}（方差）+ {n:.4f}（噪声）")
ck.ensemble_bias_variance(results_bv)

In [ ]:
# 仿 sklearn 示例的两行图：上排是真函数、5 个模型和 50 个模型的平均预测；下排是偏差²、方差、噪声随 x 的分布
x_plot = X_test_bv.ravel()
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for col, name in enumerate(["Tree", "Bagging(Tree)"]):
    P = preds_bv[name]
    ax = axes[0, col]
    for i in range(5):
        ax.plot(x_plot, P[:, i], color="C0", alpha=0.3, lw=0.8, label="5 of the 50 fitted models" if i == 0 else None)
    ax.plot(x_plot, f_test, color="k", lw=2, label="true f(x)")
    ax.plot(x_plot, P.mean(axis=1), "--", color="C3", lw=2, label="average prediction")
    ax.set_title(name)
    ax.legend(fontsize=7, loc="upper left")
    ax = axes[1, col]
    ax.plot(x_plot, (f_test - P.mean(axis=1)) ** 2, color="C3", label="bias^2(x)")
    ax.plot(x_plot, P.var(axis=1), color="C0", label="variance(x)")
    ax.plot(x_plot, y_test_bv.var(axis=1), color="C2", alpha=0.6, label="noise(x)")
    ax.set_xlabel("x")
    ax.legend(fontsize=7, loc="upper left")
axes[0, 0].set_ylabel("y")
axes[1, 0].set_ylabel("Squared error")
fig.suptitle("Bagging averages wiggly trees: same bias, much less variance")
plt.tight_layout()
plt.show()

读表和图：

- **单棵树**：偏差² 几乎为 0（0.0003）——不限深度的树把训练点全部记住，50 棵树的平均正好贴着真函数；误差的大头是**方差**（0.0153，比噪声还大）：换一份训练集，树的形状就大变（左上图那几条浅蓝色的线）。
- **Bagging**：方差降到单棵树的约 63%，偏差² 几乎不变（只多了约 0.0002）——**平均能消掉方差，消不掉偏差**。
- **随机森林**：方差再低一些（约为单棵树的 54%）。注意：这个例子只有 **1 个特征**，"每次分裂随机挑几个特征"无从挑起，随机森林在这里就是 100 棵树的 Bagging（ISLP 第 8 章的说法：Bagging 就是随机森林在"每次分裂考虑全部特征"时的特例）。它比 10 棵树的 Bagging 好，靠的是树更多。其实即使有很多特征也一样：sklearn 的 `RandomForestRegressor` 默认 `max_features=1.0`，每次分裂都看全部特征，所以回归任务上默认的"随机森林"本来就是 Bagging（S3 开头说的"默认 `sqrt`"是分类器 `RandomForestClassifier` 的默认值，分类器才会随机挑特征）；下面 California Housing 那格也是如此。
- **噪声** 0.0098 ≈ 0.1²，三个模型完全相同，谁也降不下去。三项之和与误差差一点点（单棵树 0.0254 vs 0.0256），这是只有 50 份训练集、50 份噪声带来的有限样本误差。

和 S2 的学习曲线对上：欠拟合 ≈ 高偏差，过拟合 ≈ 高方差。偏差-方差分解给了这两个词精确的定义，但只有真函数已知时才算得出来；真实数据上，我们只能靠学习曲线、验证曲线去推断。

### 偏差与方差的跷跷板：树的深度

限制树深（`max_depth`）是控制复杂度的旋钮：树越浅越"死板"（偏差高），越深越"敏感"（方差高）。把噪声加大到 σ = 0.3（数据更"脏"），单棵树从深度 1 扫到 12：

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 12]
train_sets_03, X_test_03, y_test_03 = make_datasets(noise=0.3)
sweep = []
for d in depths:
    P = np.column_stack([DecisionTreeRegressor(max_depth=d, random_state=0).fit(Xt, yt).predict(X_test_03)
                         for Xt, yt in train_sets_03])
    sweep.append(decompose(P, y_test_03, f(X_test_03)))
sweep = np.array(sweep)                                  # (9 个深度, 4 个量)

fig, ax = plt.subplots(figsize=(6, 3.5))
for j, label in enumerate(["error", "bias^2", "variance", "noise"]):
    ax.plot(depths, sweep[:, j], "o-", label=label)
ax.set_xlabel("max_depth")
ax.set_ylabel("Mean squared error")
ax.set_title(f"Noise sigma = 0.3: error is lowest at depth {depths[sweep[:, 0].argmin()]}")
ax.legend(loc="center right")
plt.show()
print("误差:", np.round(sweep[:, 0], 3))
print("偏差²:", np.round(sweep[:, 1], 4))
print("方差:", np.round(sweep[:, 2], 3))

一条 U 形曲线：深度 1–2 时偏差主导（欠拟合）；深度 3 附近两者平衡，误差最小（0.155）；再深，方差一路上升（过拟合），深度 12 时误差回到 0.19。偏差² 降到 0.0015 左右就不再降了——深度 5 以后它已经不是问题。

一个诚实的注脚：这条"教科书曲线"取决于噪声大小。把 `noise` 换回 0.1 再扫一遍（可以自己试试），方差并不随深度单调上升（深度 2 时 0.021，比不限深度的 0.015 还大），误差最小的深度也挪到了 5。分解本身永远成立，但"复杂度越高、方差越大"只是常见的趋势，不是定律。

### 另一种集成：梯度提升树

Bagging 和随机森林让很多棵深树**并排**投票，主要降方差。另一类集成反过来：**Boosting** 让很多棵浅树**排成一队**，每棵新树专门去拟合前面所有树加起来还没解释掉的部分（残差），一步步把偏差降下去。它的一个现代实现是 scikit-learn 的 `HistGradientBoostingRegressor` / `HistGradientBoostingClassifier`（下面简称 HGBT）：先把每个特征的取值分进至多 255 个箱，只在箱的边界上找分裂点，所以在上万行的数据上也很快；它还**原生支持缺失值**，不需要先填充。

在表格数据上，HGBT 是公认的强基线。在第 4 周的 California Housing 上比一比三个模型（回归任务用 `KFold` 做 5 折，指标是 R²）：

In [ ]:
import time
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

housing = fetch_california_housing(as_frame=True)
X_h, y_h = housing.data, housing.target
cv_h = KFold(n_splits=5, shuffle=True, random_state=42)

t0 = time.perf_counter()
lr_h_scores = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), X_h, y_h, cv=cv_h, scoring="r2")
t_lr = time.perf_counter() - t0
t0 = time.perf_counter()
rf_h_scores = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
                              X_h, y_h, cv=cv_h, scoring="r2")
t_rf = time.perf_counter() - t0
t0 = time.perf_counter()
# 在这里写你的代码
raise NotImplementedError
t_hgbt = time.perf_counter() - t0

for name, s, t in [("LinearRegression", lr_h_scores, t_lr), ("RandomForest", rf_h_scores, t_rf),
                   ("HistGradientBoosting", hgbt_h_scores, t_hgbt)]:
    print(f"{name:22s} R² = {s.mean():.3f} ± {s.std():.3f}   [{t:.1f} 秒]")
print("HGBT − RF 逐折:", np.round(hgbt_h_scores - rf_h_scores, 3))
ck.calhousing_models(lr_h_scores, rf_h_scores, hgbt_h_scores)

HGBT 的 R² 每一折都比随机森林高 0.02–0.03（0.836 vs 0.810），在 2 核的电脑上（例如 Colab）还快一个数量级（实测 2 线程时 RF 12.7 秒、HGBT 0.9 秒；核多的电脑上 RF 能并行，差距会小一些）；线性回归落后一大截（0.60）——房价与收入、地理位置的关系不是一条直线。scikit-learn 的示例 *Comparing Random Forests and Histogram Gradient Boosting models* 在同一份数据上得出了同样的结论：HGBT 的"速度—精度"权衡几乎总是更划算。所以从现在起，表格数据上的**强基线要同时报随机森林和 HGBT**。（上面的回归森林按默认值每次分裂看全部特征；想要真正的特征子采样，设 `max_features="sqrt"` 或 0.33。）

但别急着推广：California Housing 有 2 万行。作业 A2 会在只有 891 行的 Titanic 上再比一次，HGBT 是否更好，要你自己按逐折差值来判断。

---
## S4 · 数据泄漏：好得不像话的结果，多半是假的

**数据泄漏**＝评估时用到了"真实预测场景下不可能有"的信息。四种常见形态：

1. **目标泄漏**：特征里混进了"答案之后才会产生"的信息。例：用"是否服用了抗生素"预测"是否患肺炎"——服药发生在确诊之后。
2. **重复样本跨集合**：同一条样本既在训练集又在测试集（数据集拼接、去重不彻底时非常常见）。
3. **在全部数据上先做预处理/特征选择，再划分**：标准化、填充、选特征都"看过"测试集。
4. **同组样本跨集合**：同一位病人的多条记录、同一次实验的多个重复，被拆进了训练集和测试集。做过实验的人秒懂：这相当于把**技术重复**当成了**生物学重复**。解法是按组划分（`GroupKFold`，新版本也支持 `shuffle=True`）。

第 4 条在生物数据里最常见，也最容易被忽视。先看它在代码里长什么样：40 位病人，每人 10 条记录（技术重复）；每位病人的读数有自己的"指纹"，而标签（患病与否）与读数**毫无关系**——任何模型的真实准确率都只能约 0.5。

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(42)
pat_label = rng.permutation(np.array([0, 1] * 20))           # 40 位病人，各自一个标签，与特征无真实关系
pat_profile = rng.normal(0, 1, size=(40, 5))                 # 每位病人自己的"指纹"
groups = np.repeat(np.arange(40), 10)                        # 每人 10 条记录（技术重复）
X_g = pat_profile[groups] + rng.normal(0, 0.3, size=(400, 5))
y_g = pat_label[groups]

rf_g = RandomForestClassifier(n_estimators=100, random_state=42)
acc_by_record = cross_val_score(rf_g, X_g, y_g, cv=StratifiedKFold(5, shuffle=True, random_state=42)).mean()
acc_by_patient = cross_val_score(rf_g, X_g, y_g, cv=GroupKFold(5, shuffle=True, random_state=42),
                                 groups=groups).mean()
print(f"按记录拆（StratifiedKFold）：{acc_by_record:.3f}")
print(f"按病人拆（GroupKFold）：    {acc_by_patient:.3f}   ← 标签与特征无关，真实水平只能约 0.5")

按记录拆时，验证折里每条记录的"兄弟记录"都在训练折里，模型只要认出"这是哪位病人"就能答对——它学到的是病人的指纹，不是疾病。按病人拆（`GroupKFold`，把 `groups=` 传给 `cross_val_score`）以后，验证折里全是没见过的病人，准确率立刻回到抛硬币的水平。**有效样本量是 40 位病人，不是 400 条记录。**

只要数据里有"组"（同一病人、同一供体、同一批次、同一只小鼠、同一天的实验），就要问一句：部署时遇到的是新的组吗？是的话，就按组划分。

### 现场重演：ESL §7.10.2 的"错误的交叉验证"

《The Elements of Statistical Learning》7.10.2 节的经典实验：50 个样本、两类各半，5000 个**与标签完全无关**的标准高斯特征。任何模型的真实误差只能是 50%。现在用一个"看起来很合理"的流程：

1. 在**全部 50 个样本**上，选出与标签相关性最强的 100 个特征；
2. 只用这 100 个特征建一个 1-最近邻分类器；
3. 交叉验证估计误差。

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(42)
N, P, K = 50, 5000, 100
X_rand = rng.standard_normal((N, P))
y_rand = rng.permutation(np.array([0] * 25 + [1] * 25))


def top_corr_features(Xs, ys, k=100):
    """返回与标签相关系数绝对值最大的 k 个特征的列号"""
    Xc = Xs - Xs.mean(axis=0)
    yc = ys - ys.mean()
    corr = (Xc * yc[:, None]).sum(axis=0) / (
        np.sqrt((Xc ** 2).sum(axis=0)) * np.sqrt((yc ** 2).sum()) + 1e-12)
    return np.argsort(-np.abs(corr))[:k]

In [ ]:
# 错误流程：先在全部样本上选特征，再交叉验证
# 在这里写你的代码
raise NotImplementedError
print(f"错误流程的 CV 误差：{wrong_err:.1%}   （特征和标签明明毫无关系！）")
ck.wrong_cv_error(wrong_err)

In [ ]:
# 正确流程：特征选择放进每一折内部
# 在这里写你的代码
raise NotImplementedError
print(f"正确流程的 CV 误差：{right_err:.1%}   —— 和抛硬币一样，这才是真相")
ck.right_cv_error(right_err)

**为什么会这样？** 5000 个纯随机特征里，总有一些**碰巧**和这 50 个标签相关。在全部样本上挑出的这 100 个"幸运儿"，在留出折上也保留着这份巧合（ESL 的原文：它们与留出样本标签的相关系数平均约 0.28，而不是 0）。书中 50 次模拟的平均 CV 误差只有 **3%**——真实误差是 **50%**。

**规则（本课程的底线）**：**一切从数据里学参数的步骤**——填充用的中位数、标准化用的均值和标准差、TF-IDF 的词表和 idf（第 6 周会学）、特征选择——都放进 Pipeline，只在训练折（或训练集）上 fit。理由就是 S2 那句话：测试集代表"未来的数据"，对它的任何统计，真实使用时都不可能提前拿到。Kaggle 的泄漏教程和 scikit-learn 文档给的都是这条规则。

**它和 ESL 的说法并不矛盾。** ESL 原文的限定是：**不看标签**的无监督筛选（比如按方差筛基因）可以在划分之前做——这是统计意义上的判断：不用标签的步骤没法把"碰巧和标签相关"的巧合带进来，所以不会像上面那样把误差从 50% 压到 2%。两者的差别在于后果大小：

| 场景 | 这一步放在划分前 | 放在 CV 的每一折内部 |
| --- | --- | --- |
| Titanic：中位数填充 + 标准化（不看标签），逻辑回归 5 折准确率 | 0.789 | 0.788 |
| ESL：按与标签的相关性选特征（用到标签），1-NN 的 CV 误差 | 0.02 | 0.50 |

不看标签的步骤放错位置，在本课程的数据上影响很小（不到 0.2 个百分点）；用到标签的步骤放错位置，结论会彻底颠倒。但"很小"不等于"没有"，而且你事先无法确定它在自己的数据上有多大——所以本课程一律按底线做。记成两句话：

1. **所有 fit 都只用训练数据**（底线，全部装进 Pipeline）；
2. **用到标签的步骤放错，后果最严重。**

### 本周小结

- 逻辑回归 = 线性模型 + sigmoid；损失是交叉熵（"对正确答案的意外程度"）
- 混淆矩阵、精确率/召回率/F1/ROC-AUC；不平衡数据别只看准确率
- train/val/test；StratifiedKFold 交叉验证；报告均值 ± 标准差
- 学习曲线判断"要不要更多数据"；验证曲线调 `C`（sklearn 1.8 起别再写 `penalty`，L1 / L2 用 `l1_ratio` 选）
- 决策树是提问流程图；随机森林靠"集体投票"抗过拟合；特征重要性要用 permutation 版
- 偏差-方差分解（真函数已知时）：误差 = 偏差² + 方差 + 噪声；Bagging 与随机森林靠平均降方差，几乎不动偏差；树深是偏差与方差的跷跷板
- 梯度提升树（HGBT）一棵接一棵修残差，是表格数据的强基线；但"更强的模型"在小表格上不一定赢，要按逐折差值诚实比较
- 四类数据泄漏；同组样本要按组划分（`GroupKFold`）；**一切 fit（填充、标准化、选特征……）都只用训练折，全部装进 Pipeline**；用到标签的步骤放错，后果最严重